In [ ]:
# MLP Dataset Preparation

import os
import pandas as pd
import numpy as np

os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3,4"


In [ ]:
import torch
import random

def set_seed(seed):
    """Set random seed for reproducibility."""
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # If you use multiple GPUs
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(1)


In [ ]:
# Load quarterly data
df_weo = pd.read_csv('quarterly/df_res_light_data_sms.csv', index_col=0)
df_weo.index = [x for x in range(len(df_weo))]
df_weo


In [ ]:
# Remove rows with missing GDP values
df_weo = df_weo.dropna(subset=['GDP'])
df_weo.index = [x for x in range(len(df_weo))]


## Select Time Period

Filter the dataset to the analysis period (2013-2019).

In [ ]:
# Filter data to 2013-2019 period
df_weo = df_weo.loc[(df_weo['year'] >= 2013) & (df_weo['year'] <= 2019)]


In [ ]:
# Remove rows with any missing values
df_filter = df_weo.dropna()

# Rename columns from Chinese to English
indicator_name_map = {
    "出口金额": "export_value",
    "工业": "industry",
    "股票市值": "market_cap",
    "国际收支金融账户差额": "balance_financial_account",
    "国际收支经常账户差额": "current_account_balance",
    "国际收支经常账户贷方": "current_account_credit",
    "国际收支经常账户借方": "current_account_debit",
    "国际收支资本账户差额": "capital_account_balance",
    "国际收支资本账户贷方": "capital_account_credit",
    "国际收支资本账户借方": "capital_account_debit",
    "国际收支差额": "balance_of_payments",
    "国际投资头寸资产": "international_investment_position_assets",
    "国际投资头寸负债": "international_investment_position_liabilities",
    "国际投资头寸净额": "international_investment_position_net",
    "进口金额": "import_value",
    "名义有效汇率": "neer",
    "零售额": "retail_sales",
    "CPI": "cpi",
    "失业率": "unemployment_rate",
    "中央银行政策利率": "policy_rate",
}

# Rename columns that exist in df_filter
rename_dict = {chinese: english for chinese, english in indicator_name_map.items() if chinese in df_filter.columns}
df_filter = df_filter.rename(columns=rename_dict)

print("Columns renamed to English. Available columns:", df_filter.columns.tolist())
df_filter


In [ ]:
# Display column names (now in English)
df_filter.columns


In [ ]:
# Define feature columns using English names
data_v = [
    'export_value', 'industry', 'market_cap', 'balance_financial_account',
    'current_account_balance', 'current_account_credit', 'current_account_debit', 
    'capital_account_balance', 'capital_account_credit',
    'capital_account_debit', 'balance_of_payments', 'international_investment_position_assets', 
    'international_investment_position_liabilities', 'international_investment_position_net', 
    'import_value', 'neer', 'retail_sales', 'cpi', 'unemployment_rate', 'policy_rate',
    'sum', 'mean', 'std', 'GDP']


In [ ]:
# Define label column
label_v = ['GDP']


In [ ]:
# Normalization function: min-max scaling to [0, 1]
def norm(col):
    """Normalize column to [0, 1] range using min-max scaling."""
    return (col - col.min())/(col.max() - col.min())


In [ ]:
# Normalize all feature columns
for v in data_v:
    if v in df_filter.columns:
        df_filter[v] = df_filter[[v]].apply(lambda col: norm(col), axis=0)
    else:
        print(f"Warning: Column '{v}' not found in dataframe")


## Create Country Codes and Prepare Data

Create numeric country codes and prepare final feature/label columns for saving.

In [ ]:
# Get unique countries
country_list = df_filter['country'].unique()


In [ ]:
# Create mapping from country names to numeric codes
code_map = {}
for i in range(len(country_list)):
    code_map[country_list[i]] = i
code_map


In [ ]:
# Add country_code column to dataframe
df_filter['country_code'] = df_filter['country'].apply(lambda x: code_map[x])


In [ ]:
# Display dataframe with country codes
df_filter


In [ ]:
# Final feature columns list (including meta columns)
data_v = [
    'export_value', 'industry', 'market_cap', 'balance_financial_account',
    'current_account_balance', 'current_account_credit', 'current_account_debit', 
    'capital_account_balance', 'capital_account_credit',
    'capital_account_debit', 'balance_of_payments', 'international_investment_position_assets', 
    'international_investment_position_liabilities', 'international_investment_position_net', 
    'import_value', 'neer', 'retail_sales', 'cpi', 'unemployment_rate', 'policy_rate',
    'sum', 'mean', 'std',
    'country_code', 'year', 'quarter'
]


In [ ]:
# Label columns (GDP and meta information)
label_v = ['GDP', 'country_code', 'year', 'quarter']


In [ ]:
# Convert feature data to PyTorch tensor
data = torch.Tensor(df_filter[data_v].to_numpy())
print("Feature data shape:", data.size())
data.size()


In [ ]:
# Convert label data to PyTorch tensor
labels = torch.Tensor(df_filter[label_v].to_numpy())
print("Label data shape:", labels.size())
labels.size()


In [ ]:
# Save feature data
torch.save(data, '../dataset/MLP_data_light_sms_q_13-19.pt')
print("Saved feature data to '../dataset/MLP_data_light_sms_q_13-19.pt'")


In [ ]:
# Save label data
torch.save(labels, '../dataset/MLP_label_light_sms_q_13-19.pt')
print("Saved label data to '../dataset/MLP_label_light_sms_q_13-19.pt'")
print("\nDataset preparation complete!")
